<div align='center'>
    <h1>Implementing a Fuzzy Logic System to Evaluate Video Game Performance Based on Metadata and Platform Distribution Featur</h1>
    <h3>DKA PROJECT by STEPMTOHER LOVER</h3>
</div>

In [7]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd 

In [8]:
data_games   = pd.read_csv('C:\\Users\\ASUS\\OneDrive\\Desktop\\F.J.N Projects\\JUPYTER (AI DATABASES)\\1 STEPSISTER TUBES\\TUBES-DKA\\Dataset\\games_metadata_5k.csv')
data_ratings = pd.read_csv('C:\\Users\\ASUS\\OneDrive\\Desktop\\F.J.N Projects\\JUPYTER (AI DATABASES)\\1 STEPSISTER TUBES\\TUBES-DKA\\Dataset\\game_ratings.csv')

---
## 1. Fuzzification

In [12]:
#manual trimf helper that replicates skfuzzy's bitch ass
import numpy as np

def trimf(x, abc):
    """
    Replicates skfuzzy.trimf mathematically.
    abc is a list/tuple of three points [a, b, c].
    """
    a, b, c = abc
    
    # Handle edge cases for flat/half-triangles at boundaries
    if a == b:
        # Downward slope from b to c
        return np.where(x < b, 0.0, np.where(x <= c, (c - x) / (c - b), 0.0))
    if b == c:
        # Upward slope from a to b
        return np.where(x < a, 0.0, np.where(x <= b, (x - a) / (b - a), 0.0))
        
    # Standard triangle
    first_half = (x - a) / (b - a)
    second_half = (c - x) / (c - b)
    return np.maximum(0, np.minimum(first_half, second_half))


In [13]:
#Define the universe 
x_global = np.linspace(0, 5, 500)
x_avg_user_rating = np.linspace(1, 5, 500)
x_rating_count =np.linspace(0, 100, 500)
x_platform = np.linspace(1, 22, 500)
x_recent = np.linspace(0, 10, 500)
x_recommendation = np.linspace(0, 10, 500)

In [14]:
#Membership function 
mf_global = {
    'Poor' : trimf(x_global, 0, 0, 2.5),
    'Average' : trimf(x_global, 1.5, 2.5, 3.5),
    'High' : trimf(x_global, 3, 5, 5),
}
mf_avg_rating = {
    'Low' : trimf(x_avg_user_rating,1, 1, 3), 
    'Medium' : trimf(x_avg_user_rating,2, 3, 4), 
    'High' : trimf(x_avg_user_rating, 3, 5, 5),
}
mf_count = {
    'Few' : trimf(x_rating_count, 0, 0, 40),
    'Moderate' : trimf(x_rating_count, 20, 50, 80), 
    'Many' : trimf(x_rating_count, 60, 100, 100),
}

mf_platform = {
    'Narrow' : trimf(x_platform, 1, 1, 6), 
    'Medium' : trimf(x_platform, 3, 7, 12), 
    'Wide' : trimf(x_platform, 8, 22, 22),
}

mf_recent = {
    'Old' : trimf(x_recent, 0, 0, 4), 
    'Recent' : trimf(x_recent, 3, 5, 7), 
    'New' : trimf(x_recent, 6, 10, 10), 
}

mf_recommendation = {
    'Poor' : trimf(x_recommendation, 0, 0, 4), 
    'Fair' : trimf(x_recommendation, 3, 5, 7),
    'Excellent' : trimf(x_recommendation, 6, 10, 10),
    
}

TypeError: trimf() takes 2 positional arguments but 4 were given

In [ ]:
#Rules and stuff
def fuzzy_and(*args): 
    return min(args)
def fuzzy_or(*args): 
    return max(args)

In [ ]:
#Fuzzify helper - converts crisp inputs into membership degrees
def fuzzify(global_in, avg_rating_in, count_in, platform_in, recent_in):
    return {
        'global': {
            'Poor'    : float(trimf(global_in, [0,   0,   2.5])),
            'Average' : float(trimf(global_in, [1.5, 2.5, 3.5])),
            'High'    : float(trimf(global_in, [3,   5,   5  ])),
        },
        'avg_rating': {
            'Low'    : float(trimf(avg_rating_in, [1, 1, 3])),
            'Medium' : float(trimf(avg_rating_in, [2, 3, 4])),
            'High'   : float(trimf(avg_rating_in, [3, 5, 5])),
        },
        'count': {
            'Few'      : float(trimf(count_in, [0,  0,   40 ])),
            'Moderate' : float(trimf(count_in, [20, 50,  80 ])),
            'Many'     : float(trimf(count_in, [60, 100, 100])),
        },
        'platform': {
            'Narrow' : float(trimf(platform_in, [1, 1,  6 ])),
            'Medium' : float(trimf(platform_in, [3, 7,  12])),
            'Wide'   : float(trimf(platform_in, [8, 22, 22])),
        },
        'recent': {
            'Old'    : float(trimf(recent_in, [0, 0,  4 ])),
            'Recent' : float(trimf(recent_in, [3, 5,  7 ])),
            'New'    : float(trimf(recent_in, [6, 10, 10])),
        },
    }


#Sanity check
sample_fv = fuzzify(4.5, 4.8, 85, 18, 9)
for var, degrees in sample_fv.items():
    print(f'{var:12s}:', {k: round(v, 3) for k, v in degrees.items()})

---
## 2. Inference


In [ ]:
#Output membership shapes (Mamdani implication)
out_x         = np.linspace(0, 10, 500)
out_poor      = trimf(out_x, [0, 0,  4 ])
out_fair      = trimf(out_x, [3, 5,  7 ])
out_excellent = trimf(out_x, [6, 10, 10])

In [ ]:
#mayor mamdani

def mamdani_inference(fv):
    """
    Mamdani inference.
    fv : fuzzy values dict from fuzzify()
    Returns the aggregated output array (shape: 500,) ready for defuzzification.
    """
    g = fv['global'];  r = fv['avg_rating']
    c = fv['count'];   p = fv['platform'];  rec = fv['recent']

    # --- RULE EVALUATION & IMPLICATION ---
    # Rule 1: IF global is Poor OR avg_rating is Low THEN recommendation is Poor
    w1 = fuzzy_or(g['Poor'], r['Low'])
    rule1_clipped = np.minimum(w1, out_poor)

    # Rule 2: IF global is Average AND count is Moderate THEN recommendation is Fair
    w2 = fuzzy_and(g['Average'], c['Moderate'])
    rule2_clipped = np.minimum(w2, out_fair)

    # Rule 3: IF global is High AND recent is New AND platform is Wide THEN recommendation is Excellent
    w3 = fuzzy_and(g['High'], rec['New'], p['Wide'])
    rule3_clipped = np.minimum(w3, out_excellent)

    # --- AGGREGATION ---
    aggregated = np.maximum(rule1_clipped, np.maximum(rule2_clipped, rule3_clipped))
    return aggregated


# Example Test Run
mamdani_agg    = mamdani_inference(sample_fv)
mamdani_output = np.sum(out_x * mamdani_agg) / np.sum(mamdani_agg) if np.sum(mamdani_agg) != 0 else 5.0
print(f"Mamdani Recommendation Score: {mamdani_output:.2f} / 10")

In [ ]:
#sugenoooooooooo

def sugeno_inference(fv):
    """
    Sugeno inference.
    fv : fuzzy values dict from fuzzify()
    Returns list of (firing_strength, crisp_output) tuples ready for defuzzification.
    """
    g = fv['global'];  r = fv['avg_rating']
    c = fv['count'];   p = fv['platform'];  rec = fv['recent']

    # --- SUGENO CONSTANT CONSEQUENTS ---
    OUT_POOR      = 0.0
    OUT_FAIR      = 5.0
    OUT_EXCELLENT = 10.0

    # --- RULE EVALUATION (Firing Strengths) ---
    # Rule 1: IF global is Poor OR avg_rating is Low THEN Out = Poor
    w1 = fuzzy_or(g['Poor'], r['Low'])

    # Rule 2: IF global is Average AND count is Moderate THEN Out = Fair
    w2 = fuzzy_and(g['Average'], c['Moderate'])

    # Rule 3: IF global is High AND recent is New AND platform is Wide THEN Out = Excellent
    w3 = fuzzy_and(g['High'], rec['New'], p['Wide'])

    rules = [
        {'weight': w1, 'output': OUT_POOR},
        {'weight': w2, 'output': OUT_FAIR},
        {'weight': w3, 'output': OUT_EXCELLENT},
    ]
    return rules


# Example Test Run
sugeno_rules  = sugeno_inference(sample_fv)
numer = sum(r['weight'] * r['output'] for r in sugeno_rules)
denom = sum(r['weight'] for r in sugeno_rules)
sugeno_output = numer / denom if denom != 0 else 5.0
print(f"Sugeno Recommendation Score:  {sugeno_output:.2f} / 10")

---
## 3. Defuzzification & Evaluation


In [ ]:
print("Farrell Was Here")